# 📥 src/etl/extract.py — Interactive Walkthrough

**Project:** Nigeria Disease Surveillance Dashboard  
**Module:** `src/etl/extract.py`  
**Purpose:** Walk through every extractor function interactively,  
inspect the raw output of each source, and understand the data  
before any cleaning is applied.

---
**Functions covered:**

| Function | Source | Output |
|----------|--------|--------|
| `extract_ncdc_pdfs()` | NCDC Nigeria PDF sitreps | Raw disease tables |
| `extract_who_data()` | WHO AFRO CSV/Excel | Cross-validation data |
| `extract_nasa_rainfall()` | NASA POWER REST API | Monthly precipitation |
| `extract_health_facilities()` | HDX Nigeria CSV | Facility locations |
| `extract_population()` | NBS / WorldPop Excel | State populations |
| `extract_shapefiles()` | GRID3 Nigeria .shp | State boundaries |

---
> **Rule:** Extractors do **no cleaning**. They return data exactly as  
> found in the source. Cleaning is handled in `transform.py`.

## 1. Environment Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# Add project root to path so we can import src modules
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Paths, Diseases, settings
from src.utils.logger import get_logger, set_log_level
from src.utils.state_maps import STATE_CENTROIDS

# Show INFO logs so we can see what each extractor is doing
set_log_level('INFO')
logger = get_logger('extract_notebook')

# Make sure data directories exist
Paths.ensure_all()

print('✅ Environment ready')
print(f'   Project root  : {PROJECT_ROOT}')
print(f'   Raw data dir  : {Paths.raw}')
print(f'   Shapefiles dir: {Paths.shapefiles}')
print(f'   Diseases      : {Diseases.all}')

✅ Environment ready
   Project root  : /path/to/nigeria-disease-surveillance
   Raw data dir  : /path/to/data/raw
   Shapefiles dir: /path/to/data/shapefiles
   Diseases      : ['Cholera', 'Lassa Fever', 'Mpox', 'Meningitis', 'Yellow Fever']


## 2. Internal Helpers — `_save_raw()` and `_load_cached()`

These two private functions are used internally by every extractor.

- **`_save_raw(df, filename)`** — saves any DataFrame to `data/raw/` as CSV.  
  Called at the end of every extractor so results are cached on disk.
- **`_load_cached(filename)`** — checks whether a cached version already  
  exists in `data/raw/` and returns it, skipping a slow re-download.

This caching pattern means you only pay the download cost once.  
Pass `force_download=True` to any extractor to bypass the cache.

In [ ]:
# Import the internal helpers to inspect them
from src.etl.extract import _save_raw, _load_cached

# Demonstrate _save_raw with a tiny test DataFrame
test_df = pd.DataFrame({
    'state':   ['Lagos', 'Kano'],
    'cases':   [100, 80],
    'disease': ['Cholera', 'Cholera'],
})

saved_path = _save_raw(test_df, '_notebook_test.csv')
print(f'_save_raw() → {saved_path}')
print(f'File exists: {saved_path.exists()}')

# Demonstrate _load_cached
loaded = _load_cached('_notebook_test.csv')
print(f'\n_load_cached() → {type(loaded).__name__} with {len(loaded)} rows')
print(loaded)

# Clean up test file
saved_path.unlink()
print('\n✅ Helpers work correctly — test file cleaned up')

_save_raw() → /path/to/data/raw/_notebook_test.csv
File exists: True

_load_cached() → DataFrame with 2 rows
    state  cases  disease
0  Lagos    100  Cholera
1   Kano     80  Cholera

✅ Helpers work correctly — test file cleaned up


## 3. `extract_ncdc_pdfs()` — NCDC Situation Report PDFs

### What it does
Opens every PDF in a disease folder, finds all tables using `pdfplumber`,  
and concatenates them into one raw DataFrame.

### Key design decisions
- **Provenance columns** (`_source_file`, `_page_number`, `_table_index`)  
  are added to every row so any data issue can be traced back to its  
  exact source PDF and page.
- **No column renaming here** — column names vary across years.  
  `transform.py` handles that.
- **Bad PDFs don't crash the pipeline** — individual PDF failures are  
  logged and skipped; the remaining PDFs continue processing.

### Before running
Download NCDC sitreps from [ncdc.gov.ng/diseases/sitreps](https://ncdc.gov.ng/diseases/sitreps)  
and place them in:
```
data/raw/ncdc_pdfs/cholera/
data/raw/ncdc_pdfs/lassa_fever/
data/raw/ncdc_pdfs/mpox/
data/raw/ncdc_pdfs/meningitis/
data/raw/ncdc_pdfs/yellow_fever/
```

In [ ]:
from src.etl.extract import extract_ncdc_pdfs, _extract_single_pdf

raw_ncdc = {}

for disease_name, folder_key in Diseases.pdf_folder_map.items():
    folder = Paths.raw / 'ncdc_pdfs' / folder_key
    print(f'\n── {disease_name} ──────────────────────────────')
    print(f'   Folder: {folder}')
    print(f'   PDFs found: {len(list(folder.glob("*.pdf"))) if folder.exists() else 0}')

    df = extract_ncdc_pdfs(folder, disease_name)

    if df.empty:
        print(f'   ⚠️  No data extracted — place NCDC PDFs in the folder above')
    else:
        raw_ncdc[disease_name] = df
        print(f'   ✅ {len(df):,} rows extracted')
        print(f'   Columns : {list(df.columns)}')
        print(f'   Sources : {df["_source_file"].nunique()} unique PDFs')
        # Show the first few raw rows — expect messiness
        display(df.head(3))

print(f'\n📊 Summary: {len(raw_ncdc)}/{len(Diseases.all)} diseases extracted')

### 3.1 Inspect a single PDF

Use `_extract_single_pdf()` to inspect one specific file.  
This is useful for debugging a PDF that produced unexpected output.

In [ ]:
# Point this at any single NCDC PDF on your disk
sample_pdf = next(
    (Paths.raw / 'ncdc_pdfs' / 'cholera').glob('*.pdf'),
    None
)

if sample_pdf:
    print(f'Inspecting: {sample_pdf.name}')
    single_result = _extract_single_pdf(sample_pdf, 'Cholera')
    print(f'  Rows extracted : {len(single_result):,}')
    print(f'  Columns        : {list(single_result.columns)}')
    print(f'  Pages touched  : {single_result["_page_number"].unique()}')
    print(f'  Tables found   : {single_result["_table_index"].max() + 1}')
    display(single_result.head(5))
else:
    print('⚠️  No cholera PDFs found. Place PDFs in data/raw/ncdc_pdfs/cholera/')

## 4. `extract_who_data()` — WHO AFRO Surveillance Files

### What it does
Scans `data/raw/who/` for all `.csv` and `.xlsx` files, loads each  
one, tags it with `_source_file`, and concatenates everything.

### Why we use it
WHO AFRO data cross-validates NCDC figures. Significant discrepancies  
(> 20%) between the two sources flag a data quality issue worth investigating.

### Before running
Download from [afro.who.int](https://afro.who.int/health-topics)  
and place files in `data/raw/who/`

In [ ]:
from src.etl.extract import extract_who_data

print('Extracting WHO AFRO data...')
raw_who = extract_who_data()

if raw_who.empty:
    print('⚠️  No WHO files found in data/raw/who/')
    print('   Download from afro.who.int and place CSV/Excel files there')
else:
    print(f'✅ WHO data loaded: {len(raw_who):,} rows × {raw_who.shape[1]} columns')
    print(f'   Source files : {raw_who["_source_file"].unique()}')
    print(f'   Columns      : {list(raw_who.columns)}')
    print(f'   Null counts  :')
    nulls = raw_who.isnull().sum()
    for col, n in nulls[nulls > 0].items():
        print(f'     {col:<30} {n:>5,} ({n/len(raw_who)*100:.1f}%)')
    display(raw_who.head(3))

## 5. `extract_nasa_rainfall()` — NASA POWER API

### What it does
Queries the NASA POWER REST API for monthly precipitation (mm) at  
each of Nigeria's 37 state centroids. Builds a time series for  
rainfall-disease correlation analysis.

### API details
- **Endpoint:** `https://power.larc.nasa.gov/api/temporal/monthly/point`
- **Parameter:** `PRECTOTCORR` — precipitation corrected (mm/month)
- **Auth:** No API key required
- **Rate limit:** ~30 requests/minute → we sleep 2.2s between states
- **Fill value:** `-999.0` means NASA has no data for that month

### Internal helpers
`_fetch_one_state_rainfall()` handles one state at a time and  
returns an empty DataFrame on any error — the loop continues  
for remaining states regardless.

> ⏱️ **Expected runtime: 2–3 minutes** for all 37 states.

In [ ]:
from src.etl.extract import extract_nasa_rainfall, _fetch_one_state_rainfall

# First, test a SINGLE state before running all 37
print('Testing single-state fetch (Lagos)...')
lat, lon = 6.5244, 3.3792

test_rain = _fetch_one_state_rainfall(
    state      = 'Lagos',
    lat        = lat,
    lon        = lon,
    start_year = 2022,
    end_year   = 2023,
)

if test_rain.empty:
    print('⚠️  API call failed — check network connectivity')
    print('   The NASA POWER API requires internet access')
else:
    print(f'✅ Single-state fetch successful')
    print(f'   Rows     : {len(test_rain)}')
    print(f'   Columns  : {list(test_rain.columns)}')
    print(f'   Year range: {test_rain["year"].min()}–{test_rain["year"].max()}')
    print(f'   Avg mm/mo: {test_rain["rainfall_mm"].mean():.1f}')
    print(f'   Fill vals: {(test_rain["rainfall_mm"] == -999).sum()} rows with -999')
    display(test_rain.head(6))

In [ ]:
# Now fetch ALL 37 states (2–3 minutes)
# Set force_download=False to use a cached file if it exists
print('Fetching rainfall for all 37 states...')
print(f'State centroids configured: {len(STATE_CENTROIDS)}')
print('(Using cache if data/raw/rainfall_raw.csv exists)\n')

raw_rainfall = extract_nasa_rainfall(
    start_year     = 2015,
    end_year       = 2024,
    force_download = False,
)

if raw_rainfall.empty:
    print('⚠️  No rainfall data retrieved')
else:
    print(f'\n✅ Rainfall extraction complete')
    print(f'   Total rows     : {len(raw_rainfall):,}')
    print(f'   States covered : {raw_rainfall["state"].nunique()}/37')
    print(f'   Year range     : {raw_rainfall["year"].min()}–{raw_rainfall["year"].max()}')
    fill_count = (raw_rainfall['rainfall_mm'] == -999).sum()
    print(f'   Fill values    : {fill_count:,} (-999 → will be replaced with NaN in transform)')

    # Spot-check: verify we got 12 months × N years per state
    months_per_state = raw_rainfall.groupby('state').size()
    expected_months  = 12 * (2024 - 2015 + 1)  # 120
    short_states = months_per_state[months_per_state < expected_months]
    if short_states.empty:
        print(f'   All states have {expected_months} months of data ✅')
    else:
        print(f'   ⚠️  {len(short_states)} states have fewer than {expected_months} months:')
        print(short_states.to_string())

    display(raw_rainfall.pivot_table(
        index='month', values='rainfall_mm', aggfunc='mean'
    ).rename(columns={'rainfall_mm':'avg_rainfall_mm'}).round(1))

## 6. `extract_health_facilities()` — HDX Facility Locations

### What it does
Loads the Nigeria health facilities CSV from `data/raw/health_facilities.csv`.  
The file is manually downloaded — this extractor just reads and tags it.

### What the data contains
- Facility name, type (Hospital / PHC / Clinic)
- Ownership (Federal / State / Private / Faith-based)
- LGA and state
- **Latitude and longitude** — used for the map overlay layer

### Before running
1. Go to [data.humdata.org](https://data.humdata.org)
2. Search: `Nigeria health facilities`
3. Download the CSV and save to `data/raw/health_facilities.csv`

In [ ]:
from src.etl.extract import extract_health_facilities

print('Loading health facilities...')
raw_facilities = extract_health_facilities()

if raw_facilities.empty:
    print('⚠️  health_facilities.csv not found at data/raw/health_facilities.csv')
    print('   Download from data.humdata.org (search "Nigeria health facilities")')
else:
    print(f'✅ Health facilities loaded')
    print(f'   Rows     : {len(raw_facilities):,}')
    print(f'   Columns  : {list(raw_facilities.columns)}')

    # Check for lat/lon columns — essential for mapping
    lat_col = next((c for c in raw_facilities.columns if 'lat' in c.lower()), None)
    lon_col = next((c for c in raw_facilities.columns
                    if 'lon' in c.lower() or 'lng' in c.lower()), None)
    print(f'   Lat col  : {lat_col}  Lon col: {lon_col}')

    # Null analysis
    print(f'\n   Null counts:')
    nulls = raw_facilities.isnull().sum()
    for col, n in nulls[nulls > 0].items():
        print(f'     {col:<30} {n:>6,} ({n/len(raw_facilities)*100:.1f}%)')

    # Facility type distribution
    type_col = next(
        (c for c in raw_facilities.columns
         if 'type' in c.lower() or 'category' in c.lower()), None
    )
    if type_col:
        print(f'\n   Facility types (top 10):')
        print(raw_facilities[type_col].value_counts().head(10).to_string(indent=4))

    display(raw_facilities.head(3))

## 7. `extract_population()` — NBS / WorldPop State Populations

### What it does
Loads population estimates from an Excel or CSV file.  
Supports both `.xlsx` and `.csv` — tries Excel first, then CSV.

### Why we need it
Raw case counts are misleading without population context.  
A state with 1,000 cholera cases in a population of 5 million  
has a very different risk profile than one with 1,000 cases  
in 500,000 people.  
The incidence rate (`cases / population × 100,000`) normalises this.

### Before running
Download from [nigerianstat.gov.ng](https://nigerianstat.gov.ng)  
or [worldpop.org](https://worldpop.org) and save as  
`data/raw/nigeria_population.xlsx`

In [ ]:
from src.etl.extract import extract_population

print('Loading population data...')
raw_population = extract_population()

if raw_population.empty:
    print('⚠️  Population file not found.')
    print('   Expected: data/raw/nigeria_population.xlsx or .csv')
    print('   Download from nigerianstat.gov.ng')
else:
    print(f'✅ Population data loaded')
    print(f'   Rows     : {len(raw_population)}')
    print(f'   Columns  : {list(raw_population.columns)}')
    print(f'   Source   : {raw_population["_source_file"].iloc[0]}')

    # The transform step will identify the correct columns automatically,
    # but let's preview what we have
    print(f'\n   Preview:')
    display(raw_population.head(5))

    # Identify which column looks like population numbers
    for col in raw_population.columns:
        if col == '_source_file': continue
        sample = raw_population[col].dropna().head(3).tolist()
        print(f'   {col:<35} sample: {sample}')

## 8. `extract_shapefiles()` — GRID3 Boundary Files

### What it does
Loads state (and optionally LGA) boundary shapefiles into  
GeoDataFrames. Automatically reprojects to **WGS84 (EPSG:4326)**  
if the source CRS differs — required for PostGIS and Leaflet maps.

### What is WGS84?
WGS84 is the coordinate system used by GPS. Latitude/longitude  
values in decimal degrees. PostGIS, Folium, and every web mapping  
library expects coordinates in this system.

### Before running
Download from [grid3.org](https://grid3.org) or  
[gadm.org/country/NGA](https://gadm.org/country/NGA)  
and extract the `.shp` (+ `.shx`, `.dbf`, `.prj`) files into  
`data/shapefiles/nigeria_states.shp`

In [ ]:
from src.etl.extract import extract_shapefiles

print('Loading shapefiles...')
shapefiles = extract_shapefiles()

if not shapefiles:
    print('⚠️  No shapefiles found in data/shapefiles/')
    print('   Download from grid3.org and place nigeria_states.shp there')
else:
    for name, gdf in shapefiles.items():
        print(f'\n  ✅ {name}')
        print(f'     Features  : {len(gdf)}')
        print(f'     CRS       : {gdf.crs}')
        print(f'     EPSG:4326 : {gdf.crs.to_epsg() == 4326}')
        print(f'     Columns   : {list(gdf.columns)}')
        print(f'     Geometry  : {gdf.geom_type.unique()}')
        print(f'     Bounds    : {gdf.total_bounds.round(2)}')
        display(gdf.drop(columns=['geometry']).head(3))

    # Quick plot to visually verify geometry is correct
    if 'states' in shapefiles:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Left: plain boundary plot
        shapefiles['states'].plot(
            ax=axes[0], color='#E1F5EE',
            edgecolor='#1D9E75', linewidth=0.8
        )
        axes[0].set_title('State Boundaries', fontweight='bold')
        axes[0].axis('off')

        # Right: colour by geopolitical zone if zone column exists
        from src.utils.state_maps import GEOPOLITICAL_ZONES, normalise_state_name
        gdf_plot = shapefiles['states'].copy()

        # Find state name column
        name_col = next(
            (c for c in gdf_plot.columns
             if any(k in c.lower() for k in ['statename','state','name','adm1'])),
            None
        )
        if name_col:
            gdf_plot['canonical'] = gdf_plot[name_col].apply(normalise_state_name)
            gdf_plot['zone'] = gdf_plot['canonical'].map(GEOPOLITICAL_ZONES)
            zone_colours = {
                'North-East':'#EF9F27','North-West':'#EF9F27',
                'North-Central':'#FAEEDA',
                'South-West':'#185FA5','South-South':'#85B7EB',
                'South-East':'#1D9E75'
            }
            gdf_plot['colour'] = gdf_plot['zone'].map(zone_colours).fillna('#cccccc')
            gdf_plot.plot(ax=axes[1], color=gdf_plot['colour'],
                          edgecolor='white', linewidth=0.6)
            axes[1].set_title('States by Geopolitical Zone', fontweight='bold')
            axes[1].axis('off')

        plt.suptitle('Nigeria — GRID3 Shapefiles Verification',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(
            Paths.raw / 'shapefile_verification.png',
            dpi=150, bbox_inches='tight'
        )
        plt.show()
        print('\n  ✅ Shapefile map saved → data/raw/shapefile_verification.png')

## 9. Full Extraction Run

Run all extractors together in dependency order and produce a  
summary table — exactly what the ETL pipeline does at the  
Extract stage.

In [ ]:
print('Running all extractors...')
print('='*55)

extraction_results = {}
start = time.perf_counter()

# NCDC PDFs — one per disease
for disease_name, folder_key in Diseases.pdf_folder_map.items():
    folder = Paths.raw / 'ncdc_pdfs' / folder_key
    df_ = extract_ncdc_pdfs(folder, disease_name)
    extraction_results[f'NCDC {disease_name}'] = df_

# Other sources
extraction_results['WHO AFRO']         = extract_who_data()
extraction_results['NASA Rainfall']    = raw_rainfall  # Already fetched above
extraction_results['Health Facilities']= extract_health_facilities()
extraction_results['Population']       = extract_population()

elapsed = time.perf_counter() - start

print(f'\nExtraction complete in {elapsed:.1f}s')
print(f'\n  {"Source":<28} {"Status":<10} {"Rows":>8}')
print(f'  {"-"*28} {"-"*10} {"-"*8}')

total_rows = 0
for name, df_ in extraction_results.items():
    status = '✅ OK' if not df_.empty else '⚠️  MISSING'
    rows   = len(df_) if not df_.empty else 0
    total_rows += rows
    print(f'  {name:<28} {status:<10} {rows:>8,}')

print(f'  {"─"*28} {"─"*10} {"─"*8}')
print(f'  {"TOTAL":<28} {"":<10} {total_rows:>8,}')

# Shapefiles (GeoDataFrame — not in extraction_results)
print(f'\n  Shapefiles loaded : {len(shapefiles)} ({list(shapefiles.keys())})')
print()
print('  ➡️  All raw data ready for transform.py')

## 10. Summary — What to Expect from Each Source

In [ ]:
from datetime import datetime
print('='*60)
print('  extract.py — FUNCTION REFERENCE SUMMARY')
print('='*60)
print(f'  Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()

reference = [
    ('extract_ncdc_pdfs()', 'NCDC PDFs', 'Messy raw tables with inconsistent headers'),
    ('extract_who_data()',   'WHO AFRO',  'Cleaner CSV — use for cross-validation'),
    ('extract_nasa_rainfall()','NASA API', '~2min, -999 fill values, 12×N rows/state'),
    ('extract_health_facilities()','HDX CSV','Lat/lon coords, varies by download version'),
    ('extract_population()', 'NBS/WorldPop','Excel file, find pop column heuristically'),
    ('extract_shapefiles()', 'GRID3 .shp','GeoDataFrame, auto-reproject to WGS84'),
]

print(f'  {"Function":<32} {"Source":<14} Notes')
print(f'  {"-"*32} {"-"*14} {"-"*30}')
for func, source, note in reference:
    print(f'  {func:<32} {source:<14} {note}')

print()
print('  Key design principles:')
print('  • force_download=False  → use cache if available')
print('  • force_download=True   → always re-download')
print('  • Empty DataFrame       → source unavailable (logged, not raised)')
print('  • _source_file column   → provenance on every row')
print()
print('  ➡️  Next: run transform.py to clean all this raw data')
print('='*60)